# Anomaly Detection with Deep SVDD, SAD, and DROCC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/anomaly_detection/anomaly_detection.ipynb)

This notebook demonstrates **unsupervised anomaly detection** using Ludwig's `anomaly` output feature.
The model learns a compact representation of normal data inside a hypersphere in latent space.
At inference time, each sample is scored by its squared distance to the centre of that hypersphere:
samples far from the centre are flagged as anomalies.

```
anomaly_score = ||z - c||^2
```

**What this notebook covers:**

1. Generating a synthetic sensor dataset (normal vs anomalous readings)
2. Training with **Deep SVDD** (fully unsupervised)
3. Visualising the anomaly score distribution for normal vs anomalous samples
4. Training with **Deep SAD** (semi-supervised — uses a few labeled anomalies)
5. Training with **DROCC** (robust unsupervised — adversarial perturbations prevent collapse)
6. Comparing AUC-ROC across all three methods

In [ ]:
!pip install ludwig --quiet

## Generate synthetic sensor data

In [ ]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_NORMAL = 800
N_ANOMALY = 200

# Normal samples: Gaussian near origin
normal_df = pd.DataFrame({
    'sensor_a':       RNG.normal(0.0, 1.0, N_NORMAL),
    'sensor_b':       RNG.normal(0.0, 1.0, N_NORMAL),
    'sensor_c':       RNG.normal(0.0, 1.0, N_NORMAL),
    'timestamp_hour': RNG.integers(0, 24, N_NORMAL).astype(float),
    'anomaly':        0.0,
})

# Anomalous samples: large offset from origin
anomaly_df = pd.DataFrame({
    'sensor_a':       RNG.normal(6.0, 1.0, N_ANOMALY),
    'sensor_b':       RNG.normal(6.0, 1.0, N_ANOMALY),
    'sensor_c':       RNG.normal(6.0, 1.0, N_ANOMALY),
    'timestamp_hour': RNG.integers(0, 24, N_ANOMALY).astype(float),
    'anomaly':        1.0,
})

# ---------------------------------------------------------------
# Train split: ONLY normal samples (anomaly detection is unsupervised).
# Validation / test splits: mix of normal and anomalous.
# split column: 0=train, 1=validation, 2=test
# ---------------------------------------------------------------

normal_idx = normal_df.index.tolist()
RNG.shuffle(normal_idx)
n_train = int(0.7 * len(normal_idx))
n_val   = int(0.15 * len(normal_idx))

normal_df['split'] = 2  # test by default
normal_df.loc[normal_idx[:n_train], 'split'] = 0
normal_df.loc[normal_idx[n_train:n_train + n_val], 'split'] = 1

anom_idx = anomaly_df.index.tolist()
RNG.shuffle(anom_idx)
n_val_anom = len(anom_idx) // 2
anomaly_df['split'] = 2
anomaly_df.loc[anom_idx[:n_val_anom], 'split'] = 1

# Training CSV: only normal rows
train_df = normal_df[normal_df['split'] == 0].copy()

# Test CSV: validation + test rows (normal and anomalous), used for scoring
val_test_df = pd.concat(
    [normal_df[normal_df['split'] != 0], anomaly_df],
    ignore_index=True
)

train_df.to_csv('/tmp/sensors_train.csv', index=False)
val_test_df.to_csv('/tmp/sensors_test.csv', index=False)

print(f'Train samples : {len(train_df)} (all normal)')
print(f'Test  samples : {len(val_test_df)}')
print(f'  Normal    : {(val_test_df["anomaly"] == 0).sum()}')
print(f'  Anomalous : {(val_test_df["anomaly"] == 1).sum()}')

## Train: Deep SVDD

Deep SVDD (Ruff et al., ICML 2018) is the simplest variant: it minimises the mean squared
distance of all training representations to a hypersphere centre `c`, learned during training.
Only normal samples are used for training — no anomaly labels are required.

The `nu` parameter controls the soft-boundary relaxation: `nu=0.1` allows up to 10% of
training points outside the hypersphere.

In [ ]:
import yaml
from ludwig.api import LudwigModel

config_svdd_str = """
model_type: ecd

input_features:
  - name: sensor_a
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_b
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_c
    type: number
    preprocessing:
      normalization: zscore
  - name: timestamp_hour
    type: number
    preprocessing:
      normalization: zscore

output_features:
  - name: anomaly
    type: anomaly
    loss:
      type: deep_svdd
      nu: 0.1

trainer:
  epochs: 20
  learning_rate: 0.001

combiner:
  type: concat
  fc_layers:
    - output_size: 64
    - output_size: 32
"""

config_svdd = yaml.safe_load(config_svdd_str)

model_svdd = LudwigModel(config_svdd, logging_level=30)
train_stats_svdd, _, _ = model_svdd.train(dataset=train_df)

print('Deep SVDD training complete.')

## Evaluate: score distribution

Run inference on the test set (which contains both normal and anomalous samples) and plot
the distribution of `anomaly_score_predictions`. A good model will produce clearly separated
distributions for the two classes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

preds_svdd, _ = model_svdd.predict(dataset=val_test_df)

score_col = 'anomaly_anomaly_score_predictions'
scores_svdd = preds_svdd[score_col].values
true_labels = val_test_df['anomaly'].values

auc_svdd = roc_auc_score(true_labels, scores_svdd)
print(f'Deep SVDD  AUC-ROC: {auc_svdd:.4f}')

# --- Plot score distribution ---
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(scores_svdd[true_labels == 0], bins=40, alpha=0.6, label='Normal', color='steelblue')
ax.hist(scores_svdd[true_labels == 1], bins=40, alpha=0.6, label='Anomalous', color='tomato')

ax.set_xlabel('Anomaly score  ||z - c||^2')
ax.set_ylabel('Count')
ax.set_title(f'Deep SVDD — anomaly score distribution  (AUC = {auc_svdd:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## Try Deep SAD (semi-supervised)

Deep SAD (Ruff et al., ICLR 2020) extends Deep SVDD by incorporating a small set of
**labeled anomalies** during training. Normal/unlabeled samples are pulled toward centre `c`;
labeled anomalies are pushed away. This typically improves score separation when even a
handful of anomaly examples are available.

In the dataset, set `anomaly=1` for labeled anomaly rows, `anomaly=0` for normal rows, and
`anomaly=-1` (or leave the column absent) for unlabeled rows that should be ignored by the
repulsion term.

Here we inject ~10% labeled anomalies into the training split (`labeled_anomalies_ratio=0.1`).

In [ ]:
config_sad_str = """
model_type: ecd

input_features:
  - name: sensor_a
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_b
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_c
    type: number
    preprocessing:
      normalization: zscore
  - name: timestamp_hour
    type: number
    preprocessing:
      normalization: zscore

output_features:
  - name: anomaly
    type: anomaly
    loss:
      type: deep_sad
      eta: 1.0

trainer:
  epochs: 20
  learning_rate: 0.001

combiner:
  type: concat
  fc_layers:
    - output_size: 64
    - output_size: 32
"""

config_sad = yaml.safe_load(config_sad_str)

# Inject ~10% labeled anomalies into the training set
N_LABELED = max(1, int(0.1 * len(train_df)))
labeled_anom = anomaly_df.sample(n=N_LABELED, random_state=0).copy()
labeled_anom['split'] = 0
sad_train_df = pd.concat([train_df, labeled_anom], ignore_index=True)

print(f'Deep SAD training set: {len(sad_train_df)} rows  '
      f'({N_LABELED} labeled anomalies = '
      f'{100 * N_LABELED / len(sad_train_df):.1f}%)')

model_sad = LudwigModel(config_sad, logging_level=30)
train_stats_sad, _, _ = model_sad.train(dataset=sad_train_df)

preds_sad, _ = model_sad.predict(dataset=val_test_df)
scores_sad = preds_sad[score_col].values
auc_sad = roc_auc_score(true_labels, scores_sad)
print(f'Deep SAD   AUC-ROC: {auc_sad:.4f}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores_sad[true_labels == 0], bins=40, alpha=0.6, label='Normal', color='steelblue')
ax.hist(scores_sad[true_labels == 1], bins=40, alpha=0.6, label='Anomalous', color='tomato')
ax.set_xlabel('Anomaly score  ||z - c||^2')
ax.set_ylabel('Count')
ax.set_title(f'Deep SAD — score distribution  (AUC = {auc_sad:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## Try DROCC

DROCC (Goyal et al., ICML 2020) prevents hypersphere **collapse** — a failure mode where an
expressive encoder maps all inputs to the same point, achieving a trivially low loss while
learning nothing useful. It does so by generating adversarial perturbations around each
normal training point and penalising the model if the perturbed points score as normal.

Key parameters:
- `perturbation_strength`: magnitude of adversarial perturbations (default `0.1`)
- `num_perturbation_steps`: gradient ascent steps per sample (default `5`)

In [ ]:
config_drocc_str = """
model_type: ecd

input_features:
  - name: sensor_a
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_b
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_c
    type: number
    preprocessing:
      normalization: zscore
  - name: timestamp_hour
    type: number
    preprocessing:
      normalization: zscore

output_features:
  - name: anomaly
    type: anomaly
    loss:
      type: drocc
      perturbation_strength: 0.1
      num_perturbation_steps: 5

trainer:
  epochs: 20
  learning_rate: 0.001

combiner:
  type: concat
  fc_layers:
    - output_size: 64
    - output_size: 32
"""

config_drocc = yaml.safe_load(config_drocc_str)

model_drocc = LudwigModel(config_drocc, logging_level=30)
train_stats_drocc, _, _ = model_drocc.train(dataset=train_df)

preds_drocc, _ = model_drocc.predict(dataset=val_test_df)
scores_drocc = preds_drocc[score_col].values
auc_drocc = roc_auc_score(true_labels, scores_drocc)
print(f'DROCC      AUC-ROC: {auc_drocc:.4f}')

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores_drocc[true_labels == 0], bins=40, alpha=0.6, label='Normal', color='steelblue')
ax.hist(scores_drocc[true_labels == 1], bins=40, alpha=0.6, label='Anomalous', color='tomato')
ax.set_xlabel('Anomaly score  ||z - c||^2')
ax.set_ylabel('Count')
ax.set_title(f'DROCC — score distribution  (AUC = {auc_drocc:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

The table below compares the three loss variants on the synthetic sensor dataset.
AUC-ROC close to 1.0 means the model can almost perfectly rank anomalous samples above
normal ones. The separation ratio is the mean anomaly score divided by the mean normal score:
a ratio much greater than 1 indicates clear separation in score space.

In [ ]:
summary_rows = []
for name, scores, auc in [
    ('Deep SVDD (unsupervised)',    scores_svdd,  auc_svdd),
    ('Deep SAD  (semi-supervised)', scores_sad,   auc_sad),
    ('DROCC     (robust unsup.)',   scores_drocc, auc_drocc),
]:
    normal_s = scores[true_labels == 0]
    anom_s   = scores[true_labels == 1]
    sep = anom_s.mean() / (normal_s.mean() + 1e-9)
    summary_rows.append({
        'Method':              name,
        'AUC-ROC':             round(auc, 4),
        'Mean normal score':   round(float(normal_s.mean()), 4),
        'Mean anomaly score':  round(float(anom_s.mean()), 4),
        'Separation ratio':    round(float(sep), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))